---

## **Problem Statement**

In modern AI-driven applications, it is often necessary to enhance an agent's reasoning by giving it access to real-world information (from the internet). Large Language Models like GPT-4 alone cannot access real-time data or perform complex computations. To build a truly intelligent assistant, we need to **augment the model with tools** that bridge these gaps.

---

## **Objectives**

The goal of this application is to:

1. Build a tool-using AI agent that can:
   - Search the web for current news or facts using **Google Search via SerpAPI**
2. Integrate the above tools with **GPT-4 Turbo** using **LangChain’s latest `create_tool_calling_agent` API**
3. Maintain clean, modular, and fully compatible code using the **latest LangChain best practices**
4. Run example queries to demonstrate how the agent utilizes different tools based on the query intent

---

## **Outcomes**

- A functional, intelligent assistant capable of web search and symbolic computation.
- Tool usage is decided dynamically by the agent depending on the user’s query.
- Fully integrated with `.env` file for secure API key storage.
- The agent is extensible for further tools (e.g., weather, finance APIs, etc.).

---

In [67]:
# Step 1: Imports

from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from dotenv import load_dotenv
import os
from langchain.memory import ConversationBufferMemory
import requests


In [68]:
# Additional imports for the tool
import psutil
# psutil - A cross-platform library for retrieving information on running processes and system utilization (CPU, memory, disks, network, sensors).
from datetime import datetime
# datetime - A module for manipulating dates and times, useful for logging and timestamps.
import platform
# platform - A module that provides information about the underlying platform (OS, Python version, etc.).

In [69]:
# Install the package
# !pip install ddgs -q

# What is duckduckgo-search?
# duckduckgo-search - A Python package that allows you to perform searches using the DuckDuckGo search engine.
# It is useful for retrieving search results without using the DuckDuckGo website directly.
# It is similar to the Google search API but focuses on privacy and does not track users.

# Importing the DuckDuckGo search tool
from  ddgs import DDGS

def ddgs_search(query, max_results=3): # "LangChain Python"
    with DDGS() as ddgs:
        results = []
        for r in ddgs.text(query, max_results=max_results):
            results.append({
                'title': r['title'],
                'href': r['href'],
                'body': r['body']
            })
        return results

# Calling the search function
results = ddgs_search("LangChain Python")
# print(results)
# print(f"Number of results: {len(results)}")

# Displaying the results
for result in results:
    print(f"Title: {result['title']}")
    print(f"URL: {result['href']}")
    print(f"Snippet: {result['body']}\n")
    print("-" * 80)

Title: langchain · PyPI
URL: https://pypi.org/project/langchain/
Snippet: Jan 2, 2013 · LangChain agents are built on top of LangGraph in order to provide durable execution, streaming, human-in-the-loop, persistence, and more. (You do not need to know LangGraph for basic …

--------------------------------------------------------------------------------
Title: LangChain Tutorial - GeeksforGeeks
URL: https://www.geeksforgeeks.org/data-science/langchain-tutorial/
Snippet: Mar 2, 2026 · This section introduces LangChain and explains its purpose, core features, and main modules for building LLM-powered applications. Introduction to LangChain Modules of LangChain …

--------------------------------------------------------------------------------
Title: LangChain Python Tutorial: A Complete Guide for 2026
URL: https://blog.jetbrains.com/pycharm/2026/02/langchain-tutorial-2026/
Snippet: Feb 19, 2026 · Learn how to build AI agents with LangChain in 2026 – from chatbots and document Q&A to tool

In [70]:
# !pip install openai langchain langchain-community python-dotenv -q
# langchain-community: A community-driven extension of LangChain that includes additional tools and features. It has all the third-party integrations and community-contributed modules.
# https://python.langchain.com/api_reference/community/index.html
# https://reference.langchain.com/javascript/langchain-community

In [71]:
!pip install google-search-results -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [72]:
# External API utilities from LangChain Community
from langchain_community.utilities.serpapi import SerpAPIWrapper

# What is SerpAPIWrapper?
# SerpAPIWrapper is a utility that allows you to interact with the SerpAPI service, which provides search engine results from various search engines like Google, Bing, etc.
# It is useful for retrieving search results programmatically without having to scrape search engine result pages manually.

In [73]:
# Step 2: Load API key
from dotenv import load_dotenv
import os
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
SERPAPI_API_KEY = os.getenv("SERPAPI_API_KEY")
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
NEWS_API_KEY = os.getenv("NEWS_API_KEY")
ALPHA_VANTAGE_API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")

In [74]:
# Step 3: Set up SerpAPI (Google Search wrapper)
search_tool_google = SerpAPIWrapper(serpapi_api_key=SERPAPI_API_KEY, search_engine="google")
# search_engine:
# The search engine to use for the search. Supported values are "google", "bing", "yahoo", "duckduckgo", "baidu", "yandex", and "naver". Default is "google".

In [75]:
from langchain.tools import Tool
# Tool - A class that represents a tool that can be used by the agent to perform actions or retrieve information.

In [76]:
# Step 3: Define calculator tool
@tool # Marks this function as a LangChain-compatible tool. This makes the calculator() function callable by the agent when it decides a mathematical expression should be evaluated.
def calculator(expression: str) -> float:
    """Use this tool to evaluate a basic math operation involving + - * / and bracket ()"""
    try:
        # Evaluate the expression safely
        result = eval(expression, {"__builtins__": None}, {})
        return result
    except Exception as e:
        return f"Error evaluating expression: {e}"

@tool # Marks this function as a LangChain-compatible tool. This makes the battery_percentage() function callable by the agent when it decides to check the battery status.
def battery_percentage() -> str:
    """Get the current battery percentage of the laptop."""
    battery = psutil.sensors_battery()
    if battery is None:
        return "Battery information is not available."
    if battery.percent > 80:
        return f"Battery is at {battery.percent}%, which is good."
    elif battery.percent > 60:
        return f"Battery is at {battery.percent}%, which is okay."
    elif battery.percent > 40:
        return f"Battery is at {battery.percent}%, which is low."
    else:
        return f"Battery is at {battery.percent}%, which is critical."

@tool # Marks this function as a LangChain-compatible tool. This makes the current_time() function callable by the agent when it decides to check the current time.
def current_time() -> str:
    """Get the current time in HH:MM:SS format."""
    current_time = datetime.now()
    return f"Current time: {current_time.strftime('%H:%M:%S')}"

@tool # Marks this function as a LangChain-compatible tool. This makes the today() function callable by the agent when it decides to check today's date.
def today() -> str:
    """Get today's date."""
    today = datetime.now()
    return f"Today's date: {today.strftime('%Y-%m-%d')}"

@tool # Marks this function as a LangChain-compatible tool. This makes the system_info() function callable by the agent when it decides to check system information.
def system_info() -> str:
    """Get system name and version."""
    system_name = platform.system()
    system_version = platform.version()
    return f"System name: {system_name}, Version: {system_version}"

@tool # Marks this function as a LangChain-compatible tool. This makes the cpu_usage() function callable by the agent when it decides to check CPU usage.
def cpu_usage() -> str:
    """Get the current CPU usage percentage."""
    cpu_usage = psutil.cpu_percent(interval=1)
    return f"CPU usage: {cpu_usage}%"

@tool
def get_weather(location: str) -> str:
        """Fetch current weather data for a given location using OpenWeatherMap API."""
        base_url = "http://api.openweathermap.org/data/2.5/weather"
        params = {
            'q': location,
            'appid': OPENWEATHER_API_KEY,
            'units': 'metric'
        }
        response = requests.get(base_url, params=params)
        if response.status_code == 200: # If the request is successful (status code 200)
            data = response.json() # Parse the JSON response to extract weather information
            print("=="*30) # Print == for 30 times for better readability
            print(data) # Print the entire JSON response for debugging purposes
            print("=="*30) # Print == for 30 times for better readability
            weather_description = data['weather'][0]['description']
            temperature = data['main']['temp']
            return f"The current weather in {location} is {weather_description} with a temperature of {temperature}°C."
        else:
            return f"Unable to retrieve weather data for {location}."

@tool
def get_company_info(company: str) -> str:
    """Returns a short description of a given company."""
    company_data = {
        "oracle": "Oracle provides cloud infrastructure and enterprise software solutions.",
        "openai": "OpenAI builds advanced AI systems including ChatGPT and GPT models.",
        "google": "Google is known for its search engine, advertising, and cloud services."
    }
    return company_data.get(company.lower(), "Sorry, I don't have information on that company.") # Oracle -> oracle, OpenAI -> openai, Google -> google

@tool
def calculate_tax(income: float) -> str:
    """Estimate tax based on annual income (India - simplified slab logic)."""
    if income <= 250000:
        tax = 0
    elif income <= 500000:
        tax = (income - 250000) * 0.05
    elif income <= 1000000:
        tax = (250000 * 0.05) + (income - 500000) * 0.2
    else:
        tax = (250000 * 0.05) + (500000 * 0.2) + (income - 1000000) * 0.3
    return f"Estimated tax: ₹{tax:.2f}"

@tool
def calculate_gst(base_price: float, gst_percent: float = 18.0) -> str:
    """Calculate final price including GST."""
    gst_amount = base_price * gst_percent / 100
    final_price = base_price + gst_amount
    return f"GST: ₹{gst_amount:.2f}, Final Price: ₹{final_price:.2f}"

# Add DuckDuckGo search tool
@tool
def search_tool(query: str) -> str:
    """Search the web using DuckDuckGo."""
    results = ddgs_search(query)
    if not results: # If no results are found, return a message indicating that.
        return "No results found."
    # Format the results into a readable string
    return "\n".join([f"{res['title']}: {res['href']}\n{res['body']}" for res in results])

@tool
def get_news(topic: str) -> str:
    """
    Get the latest news headlines about a specific topic.
    Use this when the user asks for recent news or current events.
    """
    url = "https://newsapi.org/v2/everything"

    params = {
        "q": topic,
        "apiKey": NEWS_API_KEY,
        "pageSize": 5, # Limit the number of news articles returned to 5 for brevity.
        "sortBy": "publishedAt", # Sort the news articles by their publication date, with the most recent ones first.
        "language": "en" # Filter the news articles to only include those in English. You can adjust this parameter based on your needs.
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        return "Unable to fetch news at the moment."

    data = response.json()
    articles = data.get("articles", [])

    if not articles:
        return f"No news found for topic: {topic}"

    news_list = []

    for article in articles[:5]:
        title = article.get("title")
        source = article.get("source", {}).get("name")
        news_list.append(f"{title} ({source})")

    return "\n".join(news_list)

@tool
def get_stock_price(ticker: str) -> str:
    """Get the current stock price for a given ticker symbol using Alpha Vantage API."""
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "GLOBAL_QUOTE",
        "symbol": ticker,
        "apikey": ALPHA_VANTAGE_API_KEY
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        try:
            price = data["Global Quote"]["05. price"] # Extract the stock price from the JSON response. The "Global Quote" key contains the stock data, and "05. price" is the specific field for the current stock price.
            return f"The current stock price of {ticker} is ${price}."
        except KeyError:
            return f"Could not retrieve stock price for {ticker}."
    else:
        return f"Unable to fetch stock price for {ticker} at the moment."

# Step 5: Wrap both tools as LangChain Tools
tools = [
    calculator,
    battery_percentage,
    current_time,
    today,
    system_info,
    cpu_usage,
    get_weather,
    get_company_info,
    calculate_tax,
    calculate_gst,
    search_tool,
    get_news,
    get_stock_price,
    Tool.from_function(
        func=search_tool_google.run, # This function will be called when the agent decides to perform a web search.
        name="google_search", # The name of the tool as it will be called in the agent's response.
        description="Use this to search the internet for current events, recent information, or news."
    )
]

In [77]:
# Step 4: Initialize the LLM
# https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.base.ChatOpenAI.html
llm = ChatOpenAI(model="gpt-4o", temperature=0.3)  # Using gpt-4o for better performance and tool calling capabilities

In [78]:
# Step 5: Define memory to maintain conversation context during the agent's execution
memory = ConversationBufferMemory(
    memory_key="chat_history",  # Key under which the conversation history will be stored
    return_messages=True  # Instructs langchain to Return the memory as a list of messages, not as single merged text block
)

# chat_history - It is not a keyword argument, but a memory_key argument. It is used to store the conversation history in the memory object. You can name it anything you want, but it is common to use "chat_history" or "history".

In [79]:
# Step 5: Create prompt template (IMPORTANT: include input & agent_scratchpad)
# prompt = ChatPromptTemplate.from_messages([
#     ("system", "You are a smart assistant. You can perform calculations, check battery status, current time, today's date, system information, and CPU usage."),
#     ("user", "{input}"),
#     MessagesPlaceholder(variable_name="agent_scratchpad")  # Required
#     # MessagesPlaceholder - A special placeholder used by LangChain agents to store: a. Intermediate reasoning steps, b. Tool calls, c. Tool outputs - ReAct Style (Thoughts, Actions, and Observations)
# ])

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a smart assistant. You can perform calculations, check battery status, current time, today's date, system information, CPU usage, google searches, and Wolfram Alpha queries. Since you are storing the chat history for the conversation, every time you generate a response, start by clearly specify whether you referred the chat history for this response or not with 'Referring chat history' or 'Not referring chat history.' and then provide your response on the next line."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

In [80]:
# Step 6: Create agent
agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)

In [81]:
# Step 7: Wrap in executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, memory=memory)


In [82]:
# Step 9: Run a query through the agent
def run_agent(query: str):
    print(f"\nUser: {query}")
    result = agent_executor.invoke({"input": query})
    print("Agent:", result["output"])

In [18]:
# Calling tools one by one
run_agent("What is the current battery percentage of my laptop?")



User: What is the current battery percentage of my laptop?


> Entering new AgentExecutor chain...

Invoking: `battery_percentage` with `{}`


Battery is at 24%, which is critical.Not referring chat history.

The current battery percentage of your laptop is 24%, which is critical.

> Finished chain.
Agent: Not referring chat history.

The current battery percentage of your laptop is 24%, which is critical.


In [19]:
run_agent("What is the current time?")



User: What is the current time?


> Entering new AgentExecutor chain...

Invoking: `current_time` with `{}`


Current time: 19:29:33Not referring chat history.

The current time is 19:29:33.

> Finished chain.
Agent: Not referring chat history.

The current time is 19:29:33.


In [20]:
run_agent("What is today's date?")



User: What is today's date?


> Entering new AgentExecutor chain...

Invoking: `today` with `{}`


Today's date: 2026-03-29Not referring chat history.

Today's date is March 29, 2026.

> Finished chain.
Agent: Not referring chat history.

Today's date is March 29, 2026.


In [21]:
run_agent("What is my system information?")



User: What is my system information?


> Entering new AgentExecutor chain...

Invoking: `system_info` with `{}`


System name: Darwin, Version: Darwin Kernel Version 25.3.0: Wed Jan 28 20:54:55 PST 2026; root:xnu-12377.91.3~2/RELEASE_ARM64_T8132Not referring chat history.

Your system information is as follows:
- System Name: Darwin
- Version: Darwin Kernel Version 25.3.0: Wed Jan 28 20:54:55 PST 2026; root:xnu-12377.91.3~2/RELEASE_ARM64_T8132

> Finished chain.
Agent: Not referring chat history.

Your system information is as follows:
- System Name: Darwin
- Version: Darwin Kernel Version 25.3.0: Wed Jan 28 20:54:55 PST 2026; root:xnu-12377.91.3~2/RELEASE_ARM64_T8132


In [26]:
run_agent("What is the current CPU usage?")



User: What is the current CPU usage?


> Entering new AgentExecutor chain...

Invoking: `cpu_usage` with `{}`


CPU usage: 23.6%Not referring chat history.

The current CPU usage is 23.6%.

> Finished chain.
Agent: Not referring chat history.

The current CPU usage is 23.6%.


In [27]:
run_agent("What is the latest update on USA-Israel vs Iran war as of 29th March 2026?")


User: What is the latest update on USA-Israel vs Iran war as of 29th March 2026?


> Entering new AgentExecutor chain...

Invoking: `google_search` with `USA-Israel vs Iran war update March 29 2026`


[{'title': 'Iran War Live Updates: Israel Strikes Tehran as Regional Diplomats Gather in Pakistan for Talks', 'link': 'https://www.nytimes.com/live/2026/03/29/world/iran-war-trump-israel-oil', 'source': 'The New York Times', 'source_logo': 'https://serpapi.com/searches/69c93097f7e99646d86da584/images/e7tNFFTug5X1quH7xY-w9DloV1kS6GDUYlj7VaaIDEw.png', 'date': '27 minutes ago', 'live': True}, {'title': 'Live updates: Iran ‘waiting’ for ground invasion, as US moves in more troops to Middle East', 'link': 'https://www.cnn.com/2026/03/29/world/live-news/iran-war-us-israel-trump', 'source': 'CNN', 'source_logo': 'https://serpapi.com/searches/69c93097f7e99646d86da584/images/-0_sGzUvQE85b_tR2VCJMcAN9aRoOChJxmHfypAdrkg.png', 'date': '33 minutes ago', 'live': True}, {'title': 'Iran war live: Pakist

In [28]:
run_agent("What is the latest update on USA-Israel vs Iran war as of 29th March 2026? Answer in 300 words plain text without any URLs in the response.")


User: What is the latest update on USA-Israel vs Iran war as of 29th March 2026? Answer in 300 words plain text without any URLs in the response.


> Entering new AgentExecutor chain...

Invoking: `google_search` with `USA-Israel vs Iran war latest update March 29 2026`


[{'title': 'Iran War Live Updates: Israel Strikes Tehran as Regional Diplomats Gather in Pakistan for Talks', 'link': 'https://www.nytimes.com/live/2026/03/29/world/iran-war-trump-israel-oil', 'source': 'The New York Times', 'source_logo': 'https://serpapi.com/searches/69c930e7526c48ea575817cf/images/3q3_oiYYItVf88Sf82lovN_o4LQfNZ8ntoAQDyjBA84.png', 'date': '28 minutes ago', 'live': True}, {'title': 'Iran war live: Pakistan, Turkiye, Egypt, Saudi seek to de-escalate', 'link': 'https://www.aljazeera.com/news/liveblog/2026/3/29/iran-war-live-houthis-attack-israel-anti-war-protesters-rally-in-tel-aviv', 'source': 'Al Jazeera', 'source_logo': 'https://serpapi.com/searches/69c930e7526c48ea575817cf/images/x-mi96I1YoTjkwgTA

In [29]:
run_agent("Hi my name is Darshan and I am a Data Scientist. I am working on a project that involves building AI agents using LangChain. Can you help me with that?")


User: Hi my name is Darshan and I am a Data Scientist. I am working on a project that involves building AI agents using LangChain. Can you help me with that?


> Entering new AgentExecutor chain...
Not referring chat history.

Hi Darshan! I'd be happy to help you with your project involving building AI agents using LangChain. Please let me know what specific aspects you need assistance with, whether it's understanding LangChain, integrating it into your project, or solving specific challenges you're facing. Feel free to provide more details so I can assist you better!

> Finished chain.
Agent: Not referring chat history.

Hi Darshan! I'd be happy to help you with your project involving building AI agents using LangChain. Please let me know what specific aspects you need assistance with, whether it's understanding LangChain, integrating it into your project, or solving specific challenges you're facing. Feel free to provide more details so I can assist you better!


In [30]:
run_agent("Explain Langchain in 2 lines")


User: Explain Langchain in 2 lines


> Entering new AgentExecutor chain...
Referring chat history.

LangChain is a framework designed for developing applications powered by language models, enabling seamless integration and orchestration of various components like LLMs, prompts, and chains. It simplifies the process of building complex AI-driven applications by providing tools and abstractions for managing interactions between language models and other data sources.

> Finished chain.
Agent: Referring chat history.

LangChain is a framework designed for developing applications powered by language models, enabling seamless integration and orchestration of various components like LLMs, prompts, and chains. It simplifies the process of building complex AI-driven applications by providing tools and abstractions for managing interactions between language models and other data sources.


In [31]:
run_agent("Do u remember my name?")


User: Do u remember my name?


> Entering new AgentExecutor chain...
Referring chat history.

Yes, your name is Darshan.

> Finished chain.
Agent: Referring chat history.

Yes, your name is Darshan.


In [35]:
run_agent("Explain the significant achievements for India from the web about the operation Sindoor conducted in May 2025 in 10 bullet points from most to least important")


User: Explain the significant achievements for India from the web about the operation Sindoor conducted in May 2025 in 10 bullet points from most to least important


> Entering new AgentExecutor chain...

Invoking: `google_search` with `Operation Sindoor May 2025 India achievements`


['A brief armed conflict between India and Pakistan began on 7 May 2025, after India launched missile strikes on Pakistan, in a military campaign codenamed ...', 'The India-Pakistan conflict that played out between May 6 and May 10, 2025, offers several military lessons. This article presents key ...', 'The Indian Army highlighted ten major milestones of 2025, a year marked by combat operations, rapid modernisation, and a focus on precision ...', 'Once more unto the breach, India struck inside Pakistan in response to a terrorist attack. Once more, the two sides escalated — again to ...', 'India struck multiple targets in Pakistan and Pakistan-administered Kashmir in the aftermath of the Pahalgam attack.

In [36]:
run_agent("Langchain framework overview using duckduckgo search")


User: Langchain framework overview using duckduckgo search


> Entering new AgentExecutor chain...

Invoking: `search_tool` with `{'query': 'LangChain framework overview'}`


Introduction to LangChain - GeeksforGeeks: https://www.geeksforgeeks.org/artificial-intelligence/introduction-to-langchain/
Dec 12, 2025 · LangChain is an open-source framework designed to simplify the creation of applications using large language models (LLMs). It provides a standard interface for integrating with other tools …
LangChain Overview: https://deepwiki.com/langchain-ai/langchain
Mar 14, 2026 · LangChain is a modular framework for building applications with Large Language Models (LLMs) through composability. The repository implements a foundational abstraction layer that …
What is LangChain ? - IBM: https://www.ibm.com/think/topics/langchain
LangChain is an open source orchestration framework for the development of applications using large language models (LLMs), like chatbots and virtual agents.Refer

In [55]:
run_agent("What is the current weather in Mumbai?")


User: What is the current weather in Mumbai?


> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'location': 'Mumbai'}`


{'coord': {'lon': 72.8479, 'lat': 19.0144}, 'weather': [{'id': 711, 'main': 'Smoke', 'description': 'smoke', 'icon': '50n'}], 'base': 'stations', 'main': {'temp': 28.99, 'feels_like': 30.72, 'temp_min': 28.99, 'temp_max': 28.99, 'pressure': 1009, 'humidity': 58, 'sea_level': 1009, 'grnd_level': 1008}, 'visibility': 4000, 'wind': {'speed': 5.14, 'deg': 310}, 'clouds': {'all': 20}, 'dt': 1774793223, 'sys': {'type': 1, 'id': 9052, 'country': 'IN', 'sunrise': 1774746320, 'sunset': 1774790484}, 'timezone': 19800, 'id': 1275339, 'name': 'Mumbai', 'cod': 200}
The current weather in Mumbai is smoke with a temperature of 28.99°C.Not referring chat history.

The current weather in Mumbai is smoke with a temperature of 28.99°C.

> Finished chain.
Agent: Not referring chat history.

The current weather in Mumbai is smoke with a temperature of 28.99°C.


In [56]:
run_agent("What is the current weather in Bangalore?")


User: What is the current weather in Bangalore?


> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'location': 'Bangalore'}`


{'coord': {'lon': 77.6033, 'lat': 12.9762}, 'weather': [{'id': 802, 'main': 'Clouds', 'description': 'scattered clouds', 'icon': '03n'}], 'base': 'stations', 'main': {'temp': 27.02, 'feels_like': 28.18, 'temp_min': 25.18, 'temp_max': 27.23, 'pressure': 1010, 'humidity': 61, 'sea_level': 1010, 'grnd_level': 913}, 'visibility': 5000, 'wind': {'speed': 9.39, 'deg': 146, 'gust': 21.01}, 'clouds': {'all': 40}, 'dt': 1774793102, 'sys': {'type': 2, 'id': 2105374, 'country': 'IN', 'sunrise': 1774745275, 'sunset': 1774789246}, 'timezone': 19800, 'id': 1277333, 'name': 'Bengaluru', 'cod': 200}
The current weather in Bangalore is scattered clouds with a temperature of 27.02°C.Not referring chat history.

The current weather in Bangalore is scattered clouds with a temperature of 27.02°C.

> Finished chain.
Agent: Not referring chat history.

The curre

In [58]:
run_agent("What is the current weather in New York, USA?")


User: What is the current weather in New York, USA?


> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'location': 'New York, USA'}`


{'coord': {'lon': -74.006, 'lat': 40.7143}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01d'}], 'base': 'stations', 'main': {'temp': 6.6, 'feels_like': 4.05, 'temp_min': 5.25, 'temp_max': 7.97, 'pressure': 1036, 'humidity': 54, 'sea_level': 1036, 'grnd_level': 1034}, 'visibility': 10000, 'wind': {'speed': 3.6, 'deg': 170}, 'clouds': {'all': 0}, 'dt': 1774793553, 'sys': {'type': 1, 'id': 4610, 'country': 'US', 'sunrise': 1774781066, 'sunset': 1774826212}, 'timezone': -14400, 'id': 5128581, 'name': 'New York', 'cod': 200}
The current weather in New York, USA is clear sky with a temperature of 6.6°C.Not referring chat history.

The current weather in New York, USA is clear sky with a temperature of 6.6°C.

> Finished chain.
Agent: Not referring chat history.

The current weather in New York, USA is cl

In [59]:
run_agent("What is the per litre petrol price in Hyderabad today as of 2026-Mar-29? Divide the obtained per litre price by 85.")


User: What is the per litre petrol price in Hyderabad today as of 2026-Mar-29? Divide the obtained per litre price by 85.


> Entering new AgentExecutor chain...

Invoking: `search_tool` with `{'query': 'per litre petrol price in Hyderabad on 2026-Mar-29'}`


Petrol Price in Hyderabad , Check Today's Hyderabad Petrol Rates...: https://www.businesstoday.in/fuel-price/petrol-price-in-hyderabad-today
As on Mar 28, 2026 , the petrol price in Hyderabad stands at ₹107.50.On February 28, 2026 , the price of petrol in Hyderabad has decreased by ₹0.04 and the price has reached ₹107.46 per litre .
Petrol Price Today ( 29 th Mar , 2026 ), Petrol Rate in India - Goodreturns: https://www.goodreturns.in/petrol-price.html
29 th Mar , 2026 . India's petrol prices are influenced by a combination of international crude oil prices , domestic tax structures, and currency exchange rates.On March 29 , 2026 , the cost of 1 litre of petrol in Mumbai, India, is ₹103.54. Latest Updates on Petrol Price .
Petrol

In [60]:
run_agent("What are the latest AI news?")


User: What are the latest AI news?


> Entering new AgentExecutor chain...

Invoking: `get_news` with `{'topic': 'AI'}`


Unable to fetch news at the moment.Not referring chat history.

I'm unable to fetch the latest AI news at the moment. Please try again later.

> Finished chain.
Agent: Not referring chat history.

I'm unable to fetch the latest AI news at the moment. Please try again later.


In [61]:
run_agent("Give me latest news about OpenAI")


User: Give me latest news about OpenAI


> Entering new AgentExecutor chain...

Invoking: `get_news` with `{'topic': 'OpenAI'}`


Unable to fetch news at the moment.Not referring chat history.

I'm unable to fetch the latest news about OpenAI at the moment. Please try again later.

> Finished chain.
Agent: Not referring chat history.

I'm unable to fetch the latest news about OpenAI at the moment. Please try again later.


In [62]:
run_agent("What are today's world news headlines?")


User: What are today's world news headlines?


> Entering new AgentExecutor chain...

Invoking: `get_news` with `{'topic': 'world'}`


Unable to fetch news at the moment.Not referring chat history.

I'm unable to fetch today's world news headlines at the moment. Please try again later.

> Finished chain.
Agent: Not referring chat history.

I'm unable to fetch today's world news headlines at the moment. Please try again later.


In [ ]:
# alphavantage stock price
run_agent("What is the current stock price of AAPL?")

# This call will work only when the markets are open and the stock price is being updated in real-time. If you run this code outside of market hours, you may get the last available stock price or an error message depending on how the API handles requests during off-hours.

# Alternatives to alphavantage for stock price:
# 1. Yahoo Finance API (yfinance Python package)
# 2. IEX Cloud API
# 3. Finnhub API
# AMong the above, the most popular and industrially used one is Yahoo Finance API (yfinance Python package) because it is free, easy to use, and provides comprehensive stock data including historical prices, dividends, splits, and financials. It is widely adopted in the finance community for research and analysis.


User: What is the current stock price of AAPL?


> Entering new AgentExecutor chain...

Invoking: `get_stock_price` with `{'ticker': 'AAPL'}`


Could not retrieve stock price for AAPL.Not referring chat history.

I couldn't retrieve the current stock price for AAPL at the moment. Please try again later or check a financial news website for the latest information.

> Finished chain.
Agent: Not referring chat history.

I couldn't retrieve the current stock price for AAPL at the moment. Please try again later or check a financial news website for the latest information.


# Happy Learning